<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 1 &middot; Part 4 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Gradient Descent

Grid search found the bottom of the bowl by looking everywhere. Gradient descent finds it by
feeling the slope underfoot and stepping downhill. This lab implements it, tests it, watches
it run, and breaks it on purpose.

**Companion theory lecture:** CSE 815, Week 1, Session 2 (Training a Model with Gradient Descent).

## What you will be able to do

1. Implement $\partial J/\partial w$ and $\partial J/\partial b$ and **verify them numerically**.
2. Write the gradient descent loop with a correct simultaneous update.
3. Read a learning curve and say whether training worked.
4. Diagnose a learning rate that is too small, too large, or right.

---

## 1. The gradients

From the lecture, differentiating $J(w,b) = \frac{1}{2m}\sum_i (f_{w,b}(x^{(i)}) - y^{(i)})^2$:

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m}\bigl(f_{w,b}(x^{(i)}) - y^{(i)}\bigr)x^{(i)},
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\bigl(f_{w,b}(x^{(i)}) - y^{(i)}\bigr)$$

Both are averages of the residual &mdash; weighted by $x^{(i)}$ for $w$, unweighted for $b$.
That structure makes them easy to write and easy to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

x_train = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.4])
y_train = np.array([30., 36., 45., 48., 58., 62., 70., 82.])
m = x_train.shape[0]


def compute_cost(x, y, w, b):
    """Squared-error cost J(w, b)."""
    return np.sum((w * x + b - y) ** 2) / (2 * x.shape[0])


print("carried over from Part 3:  J(33, 4) =", round(compute_cost(x_train, y_train, 33.0, 4.0), 4))

In [ ]:
def compute_gradient(x, y, w, b):
    """
    Partial derivatives of the squared-error cost with respect to w and b.

    Args:
        x (ndarray (m,)): input features
        y (ndarray (m,)): true targets
        w, b (scalar)   : model parameters

    Returns:
        dj_dw (float): dJ/dw at (w, b)
        dj_db (float): dJ/db at (w, b)
    """
    m = x.shape[0]
    err = (w * x + b) - y          # the residual vector, shape (m,)
    dj_dw = np.dot(err, x) / m     # weighted by x
    dj_db = np.sum(err) / m        # unweighted
    return dj_dw, dj_db


dj_dw, dj_db = compute_gradient(x_train, y_train, 0.0, 0.0)
print(f"at (w=0, b=0):  dJ/dw = {dj_dw:.4f},  dJ/db = {dj_db:.4f}")

Both gradients are large and negative at the origin. Negative means "$J$ falls as this
parameter grows", so the update $w := w - \alpha \cdot (\text{negative})$ will *increase* $w$ —
exactly as the lecture's sign argument predicted.

---

## 2. Check the gradient before you trust it

A wrong gradient is the most common bug in machine learning, and the nastiest: the code
runs, the cost even goes down sometimes, and the model just fits badly. So test it.

The definition of a derivative gives a free reference implementation:

$$\frac{\partial J}{\partial w} \approx \frac{J(w + \varepsilon, b) - J(w - \varepsilon, b)}{2\varepsilon}$$

This **central difference** is slow &mdash; two full cost evaluations per parameter &mdash; which
is why we do not train with it. But it is derived from a completely different piece of code
than your analytic formula, so agreement between them is strong evidence both are right.

In [ ]:
def gradient_check(x, y, w, b, eps=1e-6):
    """Compare the analytic gradient with a central finite-difference estimate."""
    num_dw = (compute_cost(x, y, w + eps, b) - compute_cost(x, y, w - eps, b)) / (2 * eps)
    num_db = (compute_cost(x, y, w, b + eps) - compute_cost(x, y, w, b - eps)) / (2 * eps)
    ana_dw, ana_db = compute_gradient(x, y, w, b)
    return (ana_dw, num_dw), (ana_db, num_db)


print(f"{'point':>16} {'analytic':>12} {'numeric':>12} {'ok':>5}")
print("-" * 48)
for w_t, b_t in [(0.0, 0.0), (33.0, 4.0), (-10.0, 25.0), (60.0, -15.0)]:
    (a_w, n_w), (a_b, n_b) = gradient_check(x_train, y_train, w_t, b_t)
    print(f"{f'({w_t:g},{b_t:g}) dJ/dw':>16} {a_w:12.6f} {n_w:12.6f} "
          f"{str(np.isclose(a_w, n_w, atol=1e-4)):>5}")
    print(f"{f'({w_t:g},{b_t:g}) dJ/db':>16} {a_b:12.6f} {n_b:12.6f} "
          f"{str(np.isclose(a_b, n_b, atol=1e-4)):>5}")
    assert np.isclose(a_w, n_w, atol=1e-4) and np.isclose(a_b, n_b, atol=1e-4)

print("\nGradient check passed at every test point.")

> Run this check on every new gradient you write this semester, including the neural
> networks in Module 2. It takes two minutes and saves days.
>
> One more sanity check that costs nothing: at the minimum, both partials must be zero.
> You will confirm that at the end of this lab.

---

## 3. The gradient descent loop

$$w := w - \alpha\frac{\partial J}{\partial w}, \qquad b := b - \alpha\frac{\partial J}{\partial b}$$

**updated simultaneously.** In the code below, `compute_gradient` returns both partials
*before* either assignment happens, which is what makes the update simultaneous. Assigning
`w` first and only then computing `dj_db` would be a different (and wrong) algorithm.

In [ ]:
def gradient_descent(x, y, w_init, b_init, alpha, num_iters, record_every=1):
    """
    Run batch gradient descent on the squared-error cost.

    Args:
        x, y (ndarray (m,))      : training data
        w_init, b_init (scalar)  : starting parameters
        alpha (float)            : learning rate
        num_iters (int)          : number of iterations
        record_every (int)       : store history every this many iterations

    Returns:
        w, b (float)             : the learned parameters
        history (dict)           : 'cost', 'w', 'b' lists for plotting
    """
    w, b = float(w_init), float(b_init)
    history = {"cost": [], "w": [], "b": []}

    for i in range(num_iters):
        # Both partials are computed from the CURRENT (w, b) before either is updated.
        dj_dw, dj_db = compute_gradient(x, y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i % record_every == 0 or i == num_iters - 1:
            history["cost"].append(compute_cost(x, y, w, b))
            history["w"].append(w)
            history["b"].append(b)

    return w, b, history

In [ ]:
w_final, b_final, hist = gradient_descent(
    x_train, y_train, w_init=0.0, b_init=0.0,
    alpha=0.1, num_iters=10000, record_every=10)

print(f"learned:  w = {w_final:.4f},  b = {b_final:.4f}")
print(f"final cost J = {compute_cost(x_train, y_train, w_final, b_final):.6f}")

### An independent check on the answer

Linear regression is one of the few models with a closed-form solution, so we can confirm
gradient descent landed in the right place. `np.linalg.lstsq` solves the least-squares
problem directly.

In [ ]:
A = np.vstack([x_train, np.ones(m)]).T          # design matrix, columns [x, 1]
(w_exact, b_exact), *_ = np.linalg.lstsq(A, y_train, rcond=None)

print(f"gradient descent : w = {w_final:.6f}, b = {b_final:.6f}")
print(f"closed form      : w = {w_exact:.6f}, b = {b_exact:.6f}")
assert np.allclose([w_final, b_final], [w_exact, b_exact], atol=1e-3)
print("\nAgreement to 3 decimal places - gradient descent found the true minimum.")

# And the gradient there really is zero.
g_w, g_b = compute_gradient(x_train, y_train, w_final, b_final)
print(f"gradient at the solution: dJ/dw = {g_w:.2e}, dJ/db = {g_b:.2e}")

---

## 4. The learning curve

The single most useful diagnostic in training: cost against iteration. It must decrease.

In [ ]:
iters = np.arange(len(hist["cost"])) * 10

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

ax[0].plot(iters, hist["cost"], c="#123B63", lw=2)
ax[0].set_title("Learning curve (all iterations)")
ax[0].set_xlabel("iteration")
ax[0].set_ylabel("J(w, b)")

ax[1].plot(iters[:60], hist["cost"][:60], c="#1B7F79", lw=2, marker="o", ms=3)
ax[1].set_title("The first 600 iterations, zoomed")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel("J(w, b)")

plt.tight_layout()
plt.show()

The steep early drop followed by a long flat tail is the signature of a healthy run. The
flat part is not wasted: it is the algorithm crawling along the floor of that long, thin,
tilted valley you saw in the contour plot in Part 3.

In [ ]:
# The cost must never increase. Assert it rather than eyeballing it.
costs = np.array(hist["cost"])
increases = np.sum(np.diff(costs) > 1e-12)
print(f"iterations where J increased: {increases}")
assert increases == 0, "J increased - alpha is too large or the gradient is wrong"
print("J decreased monotonically.")

### The path taken through parameter space

In [ ]:
# Rebuild the cost surface from Part 3 so we can draw the trajectory on it.
w_grid = np.linspace(-5, 60, 140)
b_grid = np.linspace(-20, 35, 140)
W, B = np.meshgrid(w_grid, b_grid)
J_grid = np.array([[compute_cost(x_train, y_train, W[r, c], B[r, c])
                    for c in range(W.shape[1])] for r in range(W.shape[0])])

levels = np.logspace(np.log10(max(J_grid.min(), 1e-1)), np.log10(J_grid.max()), 20)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

for a, n_show, title in [(ax[0], len(hist["w"]), "The whole descent"),
                         (ax[1], 40, "The first 400 iterations")]:
    a.contour(W, B, J_grid, levels=levels, cmap="viridis", alpha=0.65)
    a.plot(hist["w"][:n_show], hist["b"][:n_show], c="#9B2C4B", lw=1.5,
           marker="o", ms=2.5, label="path")
    a.scatter([0], [0], c="#123B63", s=70, marker="s", zorder=5, label="start (0, 0)")
    a.scatter([w_final], [b_final], c="#C97B17", s=140, marker="*", zorder=6,
              label="minimum")
    a.set_xlabel("w")
    a.set_ylabel("b")
    a.set_title(title)
    a.legend(fontsize=8)

plt.tight_layout()
plt.show()

Look at the zoomed panel. The path shoots almost straight up at first, then bends and creeps
along the valley floor. It does **not** head directly for the star. The steepest-descent
direction is perpendicular to the local contour, and when contours are elongated ellipses
that direction points across the valley rather than along it. Week 2's feature scaling makes
the ellipses rounder and this path much straighter.

---

## 5. The learning rate

Now break it on purpose. Three runs, identical except for $\alpha$.

In [ ]:
runs = {}
for a in [0.001, 0.01, 0.1]:
    _, _, h = gradient_descent(x_train, y_train, 0.0, 0.0,
                               alpha=a, num_iters=1000, record_every=1)
    runs[a] = h

plt.figure(figsize=(7, 4.2))
for a, h in runs.items():
    plt.plot(h["cost"], lw=2, label=f"alpha = {a}")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("J(w, b)   (log scale)")
plt.title("Same problem, three learning rates, 1000 iterations each")
plt.legend()
plt.show()

for a, h in runs.items():
    print(f"alpha = {a:<6}  J after 1000 iterations = {h['cost'][-1]:.4f}")

`alpha = 0.001` is not wrong &mdash; it is just slow. It is heading to the same place and
will get there, in perhaps a hundred times as many iterations. On a real data set that is
the difference between a coffee break and a lost afternoon.

### Too large: divergence

For this data set the stability threshold happens to be near $\alpha \approx 0.57$. Above it
each step overshoots the minimum by more than it started away from it, and the cost explodes.

In [ ]:
_, _, h_big = gradient_descent(x_train, y_train, 0.0, 0.0,
                               alpha=0.6, num_iters=40, record_every=1)

print("cost over the first 12 iterations:")
for i, c in enumerate(h_big["cost"][:12]):
    print(f"  iter {i:2d}:  J = {c:,.2f}")

plt.figure(figsize=(6.5, 4))
plt.plot(h_big["cost"], c="#9B2C4B", lw=2, marker="o", ms=4)
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("J(w, b)   (log scale)")
plt.title("alpha = 0.6: divergence")
plt.show()

In [ ]:
# Zoom in on the first few steps in parameter space -- you can watch it overshoot.
plt.figure(figsize=(6, 5))
plt.contour(W, B, J_grid, levels=levels, cmap="viridis", alpha=0.6)
plt.plot(h_big["w"][:6], h_big["b"][:6], c="#9B2C4B", lw=1.6, marker="o", ms=5)
plt.scatter([0], [0], c="#123B63", s=70, marker="s", zorder=5, label="start")
plt.scatter([w_final], [b_final], c="#C97B17", s=140, marker="*", zorder=6, label="minimum")
plt.xlim(-5, 60)
plt.ylim(-20, 35)
plt.xlabel("w")
plt.ylabel("b")
plt.title("Each step lands further out than the last")
plt.legend(fontsize=8)
plt.show()

**The diagnostic rule, from the lecture:** if $J$ ever rises, either $\alpha$ is too large or
the gradient is wrong. You can tell the two apart: shrink $\alpha$ by a factor of ten. If the
cost now falls, it was $\alpha$. If it still rises, the bug is in `compute_gradient` &mdash;
go back to section 2.

---

## 6. Using the trained model

In [ ]:
def predict(x, w, b):
    """Prediction of the trained model at x (scalar or array)."""
    return w * x + b


print(f"trained model:  f(x) = {w_final:.2f}x + {b_final:.2f}\n")
for size in [0.9, 1.5, 2.1]:
    print(f"  {size * 1000:.0f} sq ft  ->  {predict(size, w_final, b_final):.1f} lakh BDT")

In [ ]:
x_line = np.linspace(0.6, 2.6, 50)

plt.figure(figsize=(6.5, 4.4))
plt.scatter(x_train, y_train, s=60, c="#1B7F79", zorder=3, label="Training data")
plt.plot(x_line, predict(x_line, w_final, b_final), c="#9B2C4B", lw=2, zorder=2,
         label=f"Fitted: f(x) = {w_final:.1f}x + {b_final:.1f}")
plt.scatter([1.5], [predict(1.5, w_final, b_final)], marker="D", s=90,
            c="#C97B17", zorder=4, label="Prediction at x = 1.5")
plt.title("The model gradient descent learned")
plt.xlabel("Size (1000 sq ft)")
plt.ylabel("Price (lakh BDT)")
plt.legend()
plt.show()

### Checkpoint 1

Run gradient descent from a deliberately bad start, `w_init = -40`, `b_init = 60`, with
`alpha = 0.1`. Does it reach the same `(w, b)`?

Explain your answer using the word **convex**.

In [ ]:
# Your code here

### Checkpoint 2

Write `find_best_alpha(x, y, alphas, num_iters)` that runs gradient descent for each value in
`alphas`, discards any run whose cost increased or became non-finite, and returns the
surviving $\alpha$ with the lowest final cost. Test it on
`[0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 0.6, 1.0]`.

*Hint:* `np.isfinite` catches the diverged runs.

In [ ]:
# Your code here

---

## 7. Recap

You have now built all three parts of the recipe that every algorithm in this course follows:

| Part | Piece | This week |
|---|---|---|
| Model | $f_{w,b}(x)$ | `w * x + b` |
| Cost | $J(w,b)$ | `np.sum((w*x + b - y)**2) / (2*m)` |
| Optimiser | gradient descent | `w -= alpha * dj_dw` (simultaneously with `b`) |

Everything in Modules 2 and 3 &mdash; logistic regression, neural networks, recommenders &mdash;
changes the first two rows and keeps the third.

Also worth carrying forward:

- Verify every gradient against finite differences before training with it.
- Plot the learning curve, every time. It must decrease.
- If $J$ rises: shrink $\alpha$ tenfold; if it still rises, the gradient is wrong.
- Elongated cost contours make descent zig-zag. Week 2 fixes that with feature scaling.

### Exercises to hand in

1. Reproduce the lecture's hand-worked iteration. Using $(1,1), (2,2), (3,3)$, $w=b=0$ and
   $\alpha = 0.1$, run **two** iterations and confirm your code gives $w = 0.4667$,
   $b = 0.2$ after the first. Explain why $b$ moves *away* from its optimum of 0 on
   iteration 1, even though $J$ decreases.
2. Break the simultaneous update on purpose: write `gradient_descent_sequential` that
   updates `w`, then recomputes the gradient at the new `w` before updating `b`. Plot both
   trajectories on the contour map for `alpha = 0.1`. Do they converge to the same place?
   Does that make the sequential version acceptable? Justify.
3. Add a convergence test: stop when $|J_{t-1} - J_t| < 10^{-9}$, and report the iteration
   at which it fires for `alpha` in `[0.01, 0.1, 0.3]`. Then explain why this threshold
   would be a poor choice if prices were recorded in taka rather than lakh &mdash; and propose
   a scale-invariant alternative.
4. Sabotage `compute_gradient` by dropping the `/ m` from `dj_dw` only. Predict what the
   learning curve will look like before you run it, then run it and compare. Would the
   gradient check in section 2 have caught this bug?
5. Time `gradient_descent` for 10,000 iterations against a loop-based gradient (no
   `np.dot`). Report the speed-up, and estimate the saving on a data set of $m = 10^6$.

### Next

**Week 2 &mdash; Regression with Multiple Input Variables:** several features at once,
vectorisation with `np.dot`, feature scaling to round out those contours, and a systematic
way to choose $\alpha$.

**Reading:** James et al., *ISL* 2e, &sect;3.1.1&ndash;3.1.3; Bishop, *PRML*, &sect;3.1.3
(sequential/gradient-based learning).